# Urbex: single-query grounding

Select **Runtime → Change runtime type → GPU**, then run all cells. The repository IDs below point to the current Urbex release. The agent retrieves landmarks, zooms into BEV images and inspects cached oblique views before predicting a box. The last section shows that box in an interactive 3D view.

The demo uses the released BEV and cache assets; no asset rebuilding is needed. Default 4-bit inference reduces GPU memory use and may change predictions compared with full precision.

In [ ]:
#@title Release links and demo
GITHUB_REPO = "AMAP-ML/Urbex" #@param {type:"string"}
GITHUB_REF = "main" #@param {type:"string"}
HF_DATASET_ID = "vvangfaye/Urbex-Data" #@param {type:"string"}
HF_MODEL_ID = "vvangfaye/Urbex-4B" #@param {type:"string"}
DEMO = "Car (Cambridge)" #@param ["Car (Cambridge)", "Building (Birmingham)", "Parking lot (Birmingham)"]
LOAD_IN_4BIT = True #@param {type:"boolean"}
HF_LOGIN = False #@param {type:"boolean"}

DEMOS = {
    "Car (Cambridge)": ("cityrefer_NO", 1266),
    "Building (Birmingham)": ("cityrefer_NO", 429),
    "Parking lot (Birmingham)": ("cityrefer_ND", 1007),
}
SPLIT, SAMPLE_INDEX = DEMOS[DEMO]
MAX_TURNS, MAX_NEW_TOKENS = 8, 1024

## 1. Install inference dependencies

This notebook supports Python 3.11–3.13 and uses the GPU runtime's preinstalled PyTorch/torchvision. Training dependencies are installed separately. If installation fails, the pip output below shows the underlying error.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import torch

for name, value in {"GITHUB_REPO": GITHUB_REPO, "HF_DATASET_ID": HF_DATASET_ID,
                    "HF_MODEL_ID": HF_MODEL_ID}.items():
    if "TODO" in value or len(value.split("/")) != 2:
        raise ValueError(f"Set {name} to an owner/repository ID.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime, then rerun this notebook.")

WORK_DIR = Path("/content/urbex-demo")
REPO_DIR = WORK_DIR / "repo"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(WORK_DIR / ".matplotlib")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "transformers==4.57.1", "accelerate>=0.34.2,<2", "bitsandbytes==0.48.1",
    "huggingface-hub>=0.36,<1.0", "numpy>=1.26,<2.3",
    "pillow", "scipy", "pyyaml", "plyfile", "zstandard",
])
if not REPO_DIR.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1", "--branch", GITHUB_REF,
        f"https://github.com/{GITHUB_REPO}.git", str(REPO_DIR),
    ])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GITHUB_REF])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR),
])
sys.path.insert(0, str(REPO_DIR))
print("GPU:", torch.cuda.get_device_name(0))

## 2. Download a scene

Only the selected scene’s BEV and cache archive are downloaded, verified and extracted. Original PLY is optional and is not needed for this cached-view demo. The release is public, so no token is needed; set `HF_LOGIN` only to use your own token (it is not saved in notebook outputs).

In [ ]:
import json
from pathlib import PurePosixPath
from huggingface_hub import get_token, login
from getpass import getpass
from scripts.data.download import download_metadata, download_release

if HF_LOGIN and not get_token():
    login(token=getpass("Hugging Face token (read permission): "), add_to_git_credential=False)

SOURCES = {
    "cityrefer_ND": ("cityrefer", "cityrefer_test_ND.jsonl"),
    "cityrefer_NO": ("cityrefer", "cityrefer_test_NO.jsonl"),
    "stpls3d": ("stpls3d", "stpls3d_test.jsonl"),
}
DATASET, ANNOTATION = SOURCES[SPLIT]
DOWNLOAD_DIR = WORK_DIR / "data"
DATA_ROOT, manifest, data_revision = download_metadata(HF_DATASET_ID, SPLIT, DOWNLOAD_DIR)
DATA_PATH = DATA_ROOT / "meta" / ANNOTATION
samples = [json.loads(line) for line in DATA_PATH.read_text().splitlines() if line.strip()]
if not 0 <= SAMPLE_INDEX < len(samples):
    raise IndexError(f"Sample {SAMPLE_INDEX} is outside the {len(samples)} released queries.")
sample = samples[SAMPLE_INDEX]
SCENE_ID = sample["scene_id"]
DATA_ROOT = download_release(HF_DATASET_ID, SPLIT, DOWNLOAD_DIR,
                             scenes=[SCENE_ID], revision=data_revision)
bev = PurePosixPath(sample["bev_png"])
CACHE_ROOT = DATA_ROOT / "verify_region_cache" / DATASET
required = [DATA_ROOT / bev, DATA_ROOT / bev.with_suffix(".json"),
            DATA_ROOT / f"landmarks_{DATASET}.json", CACHE_ROOT / SCENE_ID / "scene_meta.json",
            CACHE_ROOT / SCENE_ID / "regions.npy", CACHE_ROOT / SCENE_ID / "index.pkl"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing released assets: " + ", ".join(missing))
print(f"{SPLIT} · {SCENE_ID} · sample {SAMPLE_INDEX}")
print(sample["query"])


## 3. Load the released model

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.manual_seed(42)
processor = AutoProcessor.from_pretrained(HF_MODEL_ID)
load_options = {
    "dtype": DTYPE, "device_map": {"": "cuda:0"}, "attn_implementation": "sdpa",
}
if LOAD_IN_4BIT:
    load_options["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE,
    )
model = Qwen3VLForConditionalGeneration.from_pretrained(HF_MODEL_ID, **load_options).eval()
print("Loaded:", HF_MODEL_ID, "·", DTYPE, "·", "4-bit" if LOAD_IN_4BIT else "full precision")

## 4. Start the environment

The model receives the query, images and original tool schemas. Ground truth stays inside the environment for scoring.

In [ ]:
import importlib
from IPython.display import display

# Landmark paths are read at import time; BEV paths are resolved at runtime.
os.environ["CITY_AGENT_DATA"] = str(DOWNLOAD_DIR / "cityrefer")
os.environ["CITY_AGENT_DATA_STPLS3D"] = str(DOWNLOAD_DIR / "stpls3d")
os.environ["CITY_TOOL_REQUIRE_CACHE"] = "1"
from roll.pipeline.agentic.env.city_grounding import CityGroundingEnv
from roll.pipeline.agentic.env.city_grounding.tools import retrieve
importlib.reload(retrieve)  # Also allows switching demos and rerunning these cells.
os.environ["CITY_AGENT_DATA"] = str(DATA_ROOT)

env = CityGroundingEnv(
    data_path=str(DATA_PATH), mode="val", seed=42,
    multiview_cache_dir=str(CACHE_ROOT), max_tool_calls=MAX_TURNS - 1,
    max_assistant_turns=MAX_TURNS, display_max_side=768, bbox_norm_max=1000.0,
)
messages, reset_info = env.reset(seed=SAMPLE_INDEX)
tool_schemas = reset_info["tools"]
print("Query:", reset_info["query"])
display(env._env.view_image("global"))

## 5. Run the agent

Each turn uses the model's native chat template; tool observations are fed back into the next turn.

In [ ]:
def collect_images(conversation):
    return [item["image"] for message in conversation
            if isinstance(message.get("content"), list)
            for item in message["content"]
            if isinstance(item, dict) and item.get("type") == "image"]

final_info, actions = {}, []
for turn in range(1, MAX_TURNS + 1):
    prompt = processor.apply_chat_template(
        messages, tools=tool_schemas, tokenize=False, add_generation_prompt=True,
    )
    images = collect_images(messages)
    if prompt.count("<|vision_start|>") != len(images):
        raise ValueError("Chat template image placeholders do not match the tool images.")
    inputs = processor(
        text=[prompt], images=images or None, padding=True, return_tensors="pt",
    ).to(model.device)
    prompt_tokens = inputs.input_ids.shape[1]
    if prompt_tokens + MAX_NEW_TOKENS > 16384:
        raise RuntimeError("The conversation exceeds the released 16K evaluation context.")
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True,
        )
    action = processor.tokenizer.decode(generated[0, prompt_tokens:], skip_special_tokens=False)
    del inputs, generated
    print(f"\n--- Turn {turn} ---")
    print(action.replace("<|im_end|>", ""))
    prior_messages = len(messages)
    messages, reward, terminated, truncated, final_info = env.step(action)
    actions.append(action)
    for message in messages[prior_messages:]:
        if message.get("role") != "tool":
            continue
        content = message.get("content", [])
        if isinstance(content, str):
            print(content)
        else:
            for item in content:
                if item.get("type") == "image":
                    display(item["image"])
                elif item.get("type") == "text":
                    print(item["text"])
    if terminated or truncated:
        break

## 6. View the prediction in 3D

The red box is the model's predicted footprint, shown with a height measured from the scene.

In [ ]:
from IPython.display import HTML
from roll.pipeline.agentic.env.city_grounding.render import load_bev
from scripts.demo.view_3d import rebuild_scene, viewer_html

answer = final_info.get("city_answer")
world_bbox = final_info.get("city_world_bbox")
if answer and world_bbox:
    try:
        scene = rebuild_scene(CACHE_ROOT, SCENE_ID, world_bbox)
    except ValueError as error:  # for example, a box outside the cached area
        print("3D view unavailable:", error)
    else:
        # World boxes of the zoom views that lead to the answer, from the global view inwards.
        zooms, view_id = [], answer["view_id"]
        while view_id != "global":
            view = env._env.view_registry[view_id]
            zooms.insert(0, view["world_bounds"])
            view_id = view["parent_id"]
        display(HTML(viewer_html(scene, title=DEMO, subtitle=f"{SCENE_ID} · {SPLIT}", query=reset_info["query"],
                                 bev=load_bev(sample["bev_png"]), zooms=zooms)))
    print(f"BEV IoU: {final_info['city_iou_2d']:.4f}")
    print("World bbox [xmin, ymin, xmax, ymax]:", world_bbox)
else:
    print("No valid final prediction:", final_info.get("city_reason", "turn limit reached"))

result = {
    "split": SPLIT, "sample_index": SAMPLE_INDEX, "scene_id": SCENE_ID,
    "query": reset_info["query"], "model": HF_MODEL_ID, "load_in_4bit": LOAD_IN_4BIT,
    "answer": answer, "world_bbox": world_bbox, "iou_2d": final_info.get("city_iou_2d"),
    "reason": final_info.get("city_reason"), "actions": actions,
}
result_path = WORK_DIR / "result.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
print("Saved:", result_path)